In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "s3examenfinal")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_transformed = spark.table(f"{catalogo}.{esquema_source}.transformed")

percanvas,codas400,carrera,ciclo,placement,fechaplacement,tipoBD,flag_retiro,rango_edad_ingreso,rango_edad_matricula,tipo_carrera,tipo_ciclo
2263,24270571,Contabilidad,6,No Dieron PL,null,1.BD Oficial,NO,4. 24-29,4. 24-29,otros,adultos
2263,24272973,Derecho y Cienc. Polít.,10,No Dieron PL,null,1.BD Oficial,NO,5. >=30,5. >=30,otros,adultos
2263,22203338,Ing. Sistemas e Informática,6,No Dieron PL,null,1.BD Oficial,NO,1. <=18,3. 21-23,ingeniería,adultos
2263,22204103,Nutrición,9,No Dieron PL,null,1.BD Oficial,NO,1. <=18,3. 21-23,otros,adultos
2263,22204114,Derecho y Cienc. Polít.,5,No Dieron PL,null,1.BD Oficial,NO,2. 19-20,3. 21-23,otros,cachimbos
2263,23246809,Psicología,8,No Dieron PL,null,1.BD Oficial,NO,2. 19-20,3. 21-23,otros,adultos
2263,23246919,Psicología,8,No Dieron PL,null,1.BD Oficial,NO,1. <=18,2. 19-20,otros,adultos
2263,23250589,Ing. Industrial,6,No Dieron PL,null,1.BD Oficial,NO,2. 19-20,3. 21-23,ingeniería,adultos
2263,23252023,Derecho y Cienc. Polít.,9,No Dieron PL,null,1.BD Oficial,NO,5. >=30,5. >=30,otros,adultos
2263,21226269,Derecho y Cienc. Polít.,9,No Dieron PL,null,1.BD Oficial,NO,1. <=18,3. 21-23,otros,adultos


In [0]:
df_transformed = (
    df_transformed
    .groupBy(
        col("percanvas"),
        col("carrera"),
        col("tipo_ciclo"),
        col("flag_retiro"),
        col("rango_edad_ingreso"),
        col("rango_edad_matricula"),
        col("tipo_carrera")
    )
    .agg(
        count(col("codas400")).alias("conteo"),
        max(col("ciclo")).alias("max_ciclo"),
        min(col("ciclo")).alias("min_ciclo")
    )
    .orderBy(
        col("percanvas"),
        col("carrera"),
        col("tipo_ciclo"),
        col("flag_retiro"),
        col("rango_edad_ingreso"),
        col("rango_edad_matricula"),
        col("tipo_carrera").desc()
    )
)

percanvas,carrera,tipo_ciclo,flag_retiro,rango_edad_ingreso,rango_edad_matricula,tipo_carrera,conteo,max_ciclo,min_ciclo
2263,Adm. y recursos humanos,adultos,NO,4. 24-29,4. 24-29,administración,3,6,6
2263,Adm. y recursos humanos,adultos,NO,5. >=30,5. >=30,administración,2,9,6
2263,Adm. y recursos humanos,adultos,SI,5. >=30,5. >=30,administración,1,7,7
2263,Adm. y recursos humanos,cachimbos,NO,1. <=18,3. 21-23,administración,1,4,4
2263,Adm. y recursos humanos,cachimbos,NO,1. <=18,4. 24-29,administración,2,2,1
2263,Adm. y recursos humanos,cachimbos,NO,5. >=30,5. >=30,administración,1,4,4
2263,Admi. Banca y Finanzas,adultos,NO,1. <=18,2. 19-20,administración,83,9,10
2263,Admi. Banca y Finanzas,adultos,NO,1. <=18,3. 21-23,administración,158,9,10
2263,Admi. Banca y Finanzas,adultos,NO,1. <=18,4. 24-29,administración,47,9,10
2263,Admi. Banca y Finanzas,adultos,NO,2. 19-20,3. 21-23,administración,48,9,10


In [0]:
df_transformed = df_transformed.select(
    "percanvas",
    "conteo",
    "carrera",
    "max_ciclo",
    "min_ciclo",
    "flag_retiro",
    "rango_edad_ingreso",
    "rango_edad_matricula",
    "tipo_carrera",
    "tipo_ciclo"
)

In [0]:
df_transformed.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.golden")